<a href="https://colab.research.google.com/github/Ifaz2611/1719Code2024/blob/main/small_smol8btrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 14.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

compute_dtype = torch.float16

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"Using compute dtype: {compute_dtype}")

Using compute dtype: torch.float16


In [ ]:
print("Loading faster 4-bit model... Please wait.")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

model.eval()

print("\nModel loaded successfully! Type 'exit' to stop.\n" + "-" * 45)

Loading faster 4-bit model... Please wait.


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Model loaded successfully! Type 'exit' to stop.
---------------------------------------------


In [ ]:
while True:
    try:
        user_input = input("\nYou: ")

        if user_input.strip().lower() in ["exit", "quit", "q"]:
            print("Goodbye!")
            break

        if not user_input.strip():
            continue

        messages = [{"role": "user", "content": user_input}]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        print("smol: ", end="", flush=True)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

        prompt_length = inputs["input_ids"].shape[-1]

        response = tokenizer.decode(
            outputs[0][prompt_length:],
            skip_special_tokens=True
        )

        print(response)

    except KeyboardInterrupt:
        print("\nGoodbye!")
        break


You: Hola hermano
smol: Hola amigo! ¿Cómo estás? ¿Qué tal con tu estudio?

Goodbye!
